# Chapter 3

## Summarizing a document bigger than the LLM’s context window

In [1]:
with open("./Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

In [2]:
#moby_dick_book

In [3]:
#from langchain_openai import ChatOpenAI
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
#import getpass
from langchain_ollama import ChatOllama

In [4]:
#OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

In [5]:
#llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,model_name="gpt-5-nano")
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",  # opcional si es el default
    temperature=0.7,
)

In [6]:
# Split
text_chunks_chain = (
    RunnableLambda(lambda x: 
        [
            {
                'chunk': text_chunk, 
            }
            for text_chunk in 
               TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

In [7]:
# Map
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = (
    RunnableParallel (
        {
            'summary': summarize_chunk_chain | StrOutputParser()        
        }
    )
)

In [8]:
# Reduce
summarize_summaries_prompt_template = """
Write a coincise summary of the following text, which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)
summarize_reduce_chain = (
    RunnableLambda(lambda x: 
        {
            'summaries': '\n'.join([i['summary'] for i in x]), 
        })
    | summarize_summaries_prompt 
    | llm 
    | StrOutputParser()
)

In [9]:
map_reduce_chain = (
   text_chunks_chain
   | summarize_map_chain.map()
   | summarize_reduce_chain
)     

In [10]:
summary = map_reduce_chain.invoke(moby_dick_book)

In [11]:
print(summary)

Here are three concise summaries of the text with the main details:

**Summary 1**

The narrator, Ishmael, reflects on why he chose to go whaling. He cites his innate curiosity and love for the exotic, as well as a desire for adventure and exploration. Ishmael arrives in New Bedford, Massachusetts, and is disappointed that the packet ship to Nantucket has already sailed.

**Main Details:**

* Ishmael's decision to go whaling is motivated by his curiosity about the great whale and the wild seas where it lives.
* He arrives in New Bedford on a Saturday night in December and discovers that he has no time to explore the town before embarking on his voyage.
* Ishmael searches for affordable accommodations, visiting various inns and taverns.

**Summary 2**

The narrator arrives at an inn seeking shelter from a storm. The inn is filled with sailors and whalers, including a harpooneer named Bulkington. However, the narrator decides not to share a room with Bulkington and tries to find alternat

## Summarizing across documents

In [15]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader = WikipediaLoader(query="Paestum", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [16]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_docs = word_loader.load()

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_docs = pdf_loader.load()

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_docs = txt_loader.load()

In [17]:
all_docs = wikipedia_docs + word_docs + pdf_docs + txt_docs

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import getpass

In [19]:
#OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

In [20]:
#llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,model_name="gpt-5-nano")
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",  # opcional si es el default
    temperature=0.7,
)

In [21]:
refine_summary_template = """
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far: {current_refined_summary}
This is the content of the additional document: {text}
Only use the content of the additional document if it is useful, 
otherwise return the current full summary as it is."""

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)

refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [22]:
def refine_summary(docs):

    intermediate_steps = []
    current_refined_summary = ''
    for doc in docs:
        intermediate_step = \
           {"current_refined_summary": current_refined_summary, 
            "text": doc.page_content}
        intermediate_steps.append(intermediate_step)
        
        current_refined_summary = refine_chain.invoke(intermediate_step)
        
    return {"final_summary": current_refined_summary,
            "intermediate_steps": intermediate_steps}

In [ ]:
full_summary = refine_summary(all_docs)
print(full_summary)